# Flicker phosphenes: background and starter code
**Rule, Stoffregen & Ermentrout (2011), *PLoS Comput Biol* 7(9): e1002158**

This notebook gives some background on the model and a working simulator. The paper leaves out several numerical details (time step, initial condition, kernel normalization, grid units), so the choices made here are worth checking and changing.

The model, from the paper's eqs. (1)–(2), with $F(u)=1/(1+e^{-u})$, Gaussian kernels $K_{e,i}(x)=\frac{1}{\sigma_{e,i}\sqrt\pi}e^{-|x|^2/\sigma_{e,i}^2}$, periodic boundaries and stimulus $S(t)=A\,H(\sin(2\pi t/T)-0.8)$:
$$\tau_e \dot U_e = -U_e + F\big(a_{ee}K_e\star U_e - a_{ie}K_i\star U_i - \theta_e + g_e S(t)\big),\qquad
\tau_i \dot U_i = -U_i + F\big(a_{ei}K_e\star U_e - a_{ii}K_i\star U_i - \theta_i + g_i S(t)\big).$$

In [1]:
# %pip install 'pybind11>=2.12' 

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':9})

# parameters as stated in the paper's Methods
te, ti = 10., 20.                       # time constants (ms)
aee, aei, aie, aii = 10., 12., 8.5, 3.  # connection strengths
the, thi = 2., 3.5                      # thresholds
se = 2.; si = 2.5*se                    # kernel widths (here in grid cells)
F = lambda u: 1/(1+np.exp(-np.clip(u,-60,60)))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/emelierahn/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/emelierahn/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/emelierahn/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 711, in start
    self.io_loop.start()
  Fil

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## Background on the analysis
**Without the stimulus.** If $U_e$ and $U_i$ do not depend on $x$, then $K\star U$ is just $U$ (the kernels have unit integral), so the field reduces to two ordinary differential equations. The standard route is to find the equilibrium, linearize, and read the behavior near it from the eigenvalues of the Jacobian. The paper says the stable equilibrium shows damped oscillations at about 13 Hz. That frequency belongs to the oscillation around the equilibrium, not to the equilibrium itself.

**With the stimulus.** The uniform state is no longer constant. It follows a periodic orbit with the period of the flicker. To ask whether a spatial pattern can grow, the paper perturbs that orbit with a spatial Fourier mode. A Gaussian kernel simply multiplies each mode by its Fourier transform, so each mode obeys a small linear system whose coefficients are periodic in time. Stability of such a system is decided by Floquet theory. One integrates the linear system over one period to get the monodromy matrix, and the perturbation decays if all of its eigenvalues (the Floquet multipliers) lie inside the unit circle. The paper's stability curves in Figures 3 and 6 come from this calculation.

## Linear stability: the tools
(Generic background. The examples here are toys, not the phosphene model.)

**Fixed points.** For $\dot{\mathbf x}=\mathbf f(\mathbf x)$ with $\mathbf f(\mathbf x^*)=0$, put $\mathbf x=\mathbf x^*+\mathbf p$:
$$\dot{\mathbf p}=J\mathbf p+O(|\mathbf p|^2),\qquad J_{jk}=\partial f_j/\partial x_k\big|_{\mathbf x^*},\qquad \mathbf p(t)=\sum_k c_k\mathbf v_ke^{\lambda_kt},\ \ J\mathbf v_k=\lambda_k\mathbf v_k.$$
Write $\lambda=\alpha\pm i\omega$. The fixed point is stable iff all $\alpha<0$, and oscillations have frequency $\omega/2\pi$. For $2\times2$,
$$\lambda=\tfrac12\Big(\mathrm{tr}J\pm\sqrt{(\mathrm{tr}J)^2-4\det J}\Big):\quad \text{stable iff } \mathrm{tr}J<0,\ \det J>0;\quad \text{oscillatory iff } (\mathrm{tr}J)^2<4\det J,\ \ \omega=\tfrac12\sqrt{4\det J-(\mathrm{tr}J)^2}.$$

**Periodic coefficients.** If $\dot{\mathbf p}=A(t)\mathbf p$ with $A(t+T)=A(t)$, solve $\dot M=A(t)M$, $M(0)=I$, and set $M_T=M(T)$ (the *monodromy matrix*). Then $\mathbf p(nT)=M_T^{\,n}\mathbf p(0)$, so stability is decided by the eigenvalues $\mu$ of $M_T$ (the *Floquet multipliers*, $\mu=e^{\lambda T}$): decay iff all $|\mu|<1$. For $2\times2$, $\mu^2-\mathrm{tr}(M_T)\mu+\det M_T=0$ with $\det M_T=\exp\!\int_0^T\mathrm{tr}A\,dt$, and both multipliers are inside the unit circle iff $\det M_T<1$ and $|\mathrm{tr}M_T|<1+\det M_T$. They leave the unit circle at $\mu=+1$ (growth with period $T$), at $\mu=-1$ (growth with a sign flip every period, so period $2T$), or as a complex pair (a second frequency).

In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve

# Toy periodic system: damped Mathieu equation x'' + c x' + (a + q cos t) x = 0, period 2 pi
def monodromy(a, q, c=0.1, T=2*np.pi):
    def rhs(t, y):                                # y holds the 2x2 matrix M, flattened
        A = np.array([[0., 1.], [-(a + q*np.cos(t)), -c]])
        return (A @ y.reshape(2, 2)).ravel()
    sol = solve_ivp(rhs, [0, T], np.eye(2).ravel(), rtol=1e-10, atol=1e-12)
    return sol.y[:, -1].reshape(2, 2)

for a, q in [(0.5, 0.0), (0.25, 0.8), (0.25, 0.1)]:
    mu = np.linalg.eigvals(monodromy(a, q))
    print(f"a={a}, q={q}:  multipliers {mu.round(3)}   max|mu| = {abs(mu).max():.3f}   ->", "grows" if abs(mu).max() > 1 else "decays")

# a multiplier below -1 seen in time, sampled once per period
sol = solve_ivp(lambda t, y: [y[1], -0.1*y[1] - (0.25 + 0.8*np.cos(t))*y[0]], [0, 8*np.pi], [1., 0.], dense_output=True, rtol=1e-10, atol=1e-12)
print("x(nT), n=0..4:", np.array([sol.sol(n*2*np.pi)[0] for n in range(5)]).round(2), "  (each step multiplies by about -5.68)")

## Space and time together: an unforced field
**A toy field.** Two populations $\mathbf u=(u_e,u_i)$ on a line, with exponential kernels and parameters unrelated to the paper:
$$\tau_j\,\partial_tu_j=-u_j+F\Big(\sum_kW_{jk}\,(\kappa_k\star u_k)+h_j\Big),\qquad \kappa_k(y)=\tfrac{1}{2\ell_k}e^{-|y|/\ell_k}.$$
A uniform fixed point $\mathbf u^*$ solves $u_j^*=F\big(\sum_kW_{jk}u_k^*+h_j\big)$, because $\kappa\star(\text{const})=\text{const}$ when $\int\kappa=1$. The question is whether a *spatially varying* perturbation of $\mathbf u^*$ can grow.

**1. Linearize.** Put $\mathbf u=\mathbf u^*+\epsilon\mathbf p(x,t)$ and let $\mathrm{arg}^*_j=\sum_kW_{jk}u_k^*+h_j$. Then $F(\mathrm{arg}^*_j+\epsilon\delta_j)=u_j^*+\epsilon F'_j\delta_j+O(\epsilon^2)$ with $\delta_j=\sum_kW_{jk}\,\kappa_k\star p_k$, so
$$\tau_j\,\partial_tp_j=-p_j+F'_j\sum_kW_{jk}\,(\kappa_k\star p_k),\qquad F'_j=F'(\mathrm{arg}^*_j).$$
This is linear and has coefficients that do not depend on $x$, so plane waves are its natural solutions.

**2. Convolution of a plane wave.** For $\int\kappa=1$,
$$\kappa\star e^{iqx}=\int\kappa(y)\,e^{iq(x-y)}\,dy=\hat\kappa(q)\,e^{iqx},\qquad \hat\kappa(q)=\int\kappa(y)e^{-iqy}dy.$$
For the exponential kernel $\hat\kappa(q)=\int\frac{1}{2\ell}e^{-|y|/\ell}\cos(qy)\,dy=\dfrac{1}{1+\ell^2q^2}$, and for $\frac{1}{\sigma\sqrt\pi}e^{-y^2/\sigma^2}$ it is $e^{-\sigma^2q^2/4}$. Convolution has become multiplication, so each wavenumber decouples from the others.

**3. Matrix problem for each $q$.** Insert $\mathbf p=\mathbf Pe^{iqx+\lambda t}$:
$$\lambda\mathbf P=J(q)\mathbf P,\qquad J(q)=D\big[-I+G\,\hat W(q)\big],\quad D=\mathrm{diag}\Big(\tfrac1{\tau_e},\tfrac1{\tau_i}\Big),\ G=\mathrm{diag}(F'_e,F'_i),\ \hat W_{jk}(q)=W_{jk}\hat\kappa_k(q).$$
Each $q$ gives a $2\times2$ matrix with two eigenvalues $\lambda_\pm(q)$. The curve $\mathrm{Re}\,\lambda_{\max}(q)$ is the **dispersion relation**.

**4. Reading the dispersion relation.**

| what you see | meaning |
|---|---|
| $\mathrm{Re}\,\lambda(q)<0$ for all $q$ | uniform state is stable to every pattern |
| $\mathrm{Re}\,\lambda>0$ on a band around $q^*\neq0$, $\mathrm{Im}\,\lambda(q^*)=0$ | stationary pattern, wavelength $2\pi/q^*$ (Turing) |
| same, but $\mathrm{Im}\,\lambda(q^*)\neq0$ | pattern that oscillates or travels (wave/Hopf) |
| $\mathrm{Re}\,\lambda>0$ only at $q=0$ | the uniform state itself is unstable (no pattern) |

In terms of the $2\times2$ matrix: a real eigenvalue crosses zero when $\det J(q)=0$ (stationary), and a complex pair crosses when $\mathrm{tr}J(q)=0$ with $\det J(q)>0$ (oscillating).

**5. Finite domains.** On a periodic domain of length $L$ the allowed wavenumbers are $q=2\pi n/L$, so a band of unstable $q$ only matters if it contains one of these. In two dimensions $\mathbf q=\frac{2\pi}{L}(n,m)$ and for radially symmetric kernels $J$ depends only on $|\mathbf q|$, so every $(n,m)$ with the same $n^2+m^2$ grows at the same rate. Which combination of them wins is decided by nonlinear effects and the initial condition, not by the linear calculation.

In [ ]:
# Toy field parameters (not the paper's)
W   = np.array([[14., -10.], [12., -3.]]); h = np.array([-3., -4.]); tau = np.array([1., 1.]); ell = np.array([1., 3.])
D   = np.diag(1/tau); khat = lambda q: 1/(1+(ell*q)**2)               # khat[k]: one entry per source population
What = lambda q: W*khat(q)[None, :]
Fn  = lambda u: 1/(1+np.exp(-u))

ustar = fsolve(lambda u: -u + Fn(W@u + h), [.3, .3]); a = W@ustar + h; G = np.diag(Fn(a)*(1-Fn(a)))
Jq  = lambda q: D@(-np.eye(2) + G@What(q))
qs  = np.linspace(0, 3, 301)
lam = np.array([max(np.linalg.eigvals(Jq(q)), key=lambda z: z.real) for q in qs])
tr  = np.array([np.trace(Jq(q)) for q in qs]); dt_ = np.array([np.linalg.det(Jq(q)) for q in qs])
i = lam.real.argmax()
print(f"u* = {ustar.round(3)}.  q=0:  lambda = {lam[0].round(3)},  tr = {tr[0]:.3f}, det = {dt_[0]:.3f}   (uniform state stable)")
print(f"max Re lambda = {lam.real[i]:.3f} at q* = {qs[i]:.2f}, Im = {lam.imag[i]:.2f}  ->  stationary pattern, wavelength {2*np.pi/qs[i]:.1f}")
print(f"det J(q) < 0 for q in [{qs[dt_<0].min():.2f}, {qs[dt_<0].max():.2f}], which is the unstable band")
fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.8))
ax[0].plot(qs, lam.real); ax[0].axhline(0, c='k', lw=.5); ax[0].set(xlabel='q', ylabel='Re $\\lambda_{max}(q)$')
ax[1].plot(qs, tr, label='tr J(q)'); ax[1].plot(qs, dt_, label='det J(q)'); ax[1].axhline(0, c='k', lw=.5); ax[1].set(xlabel='q'); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**On a periodic domain.** The code below lists which wavenumbers the box actually allows inside the unstable band, and how many 2D lattice vectors share the fastest-growing $|\mathbf q|$.

In [ ]:
L = 100.
band = qs[lam.real > 0]; qlo, qhi = band.min(), band.max()
n_allowed = [n for n in range(1, 40) if qlo < 2*np.pi*n/L < qhi]
print(f"L = {L:.0f}: allowed q = 2 pi n/L inside the band ({qlo:.2f}, {qhi:.2f}) for n = {n_allowed[0]}..{n_allowed[-1]}  ({len(n_allowed)} modes in 1D)")
nmax = int(qhi*L/(2*np.pi)) + 2
lattice = [(n, m) for n in range(-nmax, nmax+1) for m in range(-nmax, nmax+1) if (n, m) != (0, 0)]
rate = lambda n, m: max(np.linalg.eigvals(Jq(2*np.pi*np.hypot(n, m)/L))).real
best = max(rate(n, m) for n, m in lattice)
fast = [(n, m) for n, m in lattice if abs(rate(n, m) - best) < 1e-9]
print(f"2D: fastest |q| = {2*np.pi*np.hypot(*fast[0])/L:.3f}, growth rate {best:.3f}, shared by {len(fast)} lattice vectors, for example {fast[:4]}")

### Check the unforced theory against a simulation
Simulate the toy field on a periodic domain from a tiny cosine perturbation of $\mathbf u^*$, and fit the growth rate of the Fourier coefficient at $q=2\pi n/L$. It should match $\mathrm{Re}\,\lambda(q)$.

In [ ]:
N = 256; xg = np.arange(N)*L/N; qg = 2*np.pi*np.fft.fftfreq(N, d=L/N)
kh = np.stack([1/(1+(l*qg)**2) for l in ell])                          # kappa-hat on the grid, one row per population
def field_rhs(u, drive=0.):
    conv = np.real(np.fft.ifft(kh*np.fft.fft(u, axis=1), axis=1))        # kappa * u for each population
    return (-u + Fn(np.einsum('jk,kx->jx', W, conv) + h[:, None] + drive)) / tau[:, None]
def heun(u, dt, drive0=0., drive1=0.):
    k1 = field_rhs(u, drive0); k2 = field_rhs(u + dt*k1, drive1); return u + dt/2*(k1 + k2)

def toy_growth(n, amp=1e-6, tmax=6., dt=0.005):
    u = np.array([ustar[0] + amp*np.cos(2*np.pi*n*xg/L), ustar[1] + 0*xg]); amps, ts = [], []
    for s in range(int(tmax/dt)):
        u = heun(u, dt)
        if s % 20 == 0 and s*dt > 2.: amps.append(abs(np.fft.fft(u[0]-ustar[0])[n])); ts.append(s*dt)   # skip transient
    return np.polyfit(ts, np.log(amps), 1)[0]
print(" n     q     Re lambda(q) predicted    growth rate measured")
for n in (2, 4, 6, 8, 10, 14):
    q = 2*np.pi*n/L; print(f"{n:2d}  {q:5.2f}        {max(np.linalg.eigvals(Jq(q))).real:8.3f}                {toy_growth(n):8.3f}")

## Space and time together: a periodically forced field
Add a drive to the excitatory input, $s_e(t)=A\cos(2\pi t/T)$, so the equations become $\tau_j\partial_tu_j=-u_j+F\big(\sum_kW_{jk}\kappa_k\star u_k+h_j+s_j(t)\big)$.

**1. The base state is a periodic orbit.** Uniform solutions satisfy the ordinary differential equations
$$\tau_j\dot u_{0,j}=-u_{0,j}+F\big(\textstyle\sum_kW_{jk}u_{0,k}+h_j+s_j(t)\big),$$
and they are solutions of the full field too. Starting from $\mathbf u^*$ and integrating for several periods lets the transient die out. What remains is a $T$-periodic orbit $\mathbf u_0(t)$ (assuming it is attracting).

**2. Linearize about the orbit.** Put $\mathbf u=\mathbf u_0(t)+\epsilon\mathbf p(x,t)$. The derivation in the unforced section repeats line by line, except that $F'_j$ is now evaluated along the orbit and so depends on time:
$$\tau_j\partial_tp_j=-p_j+F'_j(t)\sum_kW_{jk}\,\kappa_k\star p_k,\qquad F'_j(t)=F'\Big(\sum_kW_{jk}u_{0,k}(t)+h_j+s_j(t)\Big).$$

**3. Fourier modes still decouple.** The coefficients depend on $t$ but not on $x$, so $\mathbf p=\mathbf P(t)e^{iqx}$ is still allowed and each $q$ gets its own system with periodic coefficients:
$$\dot{\mathbf P}=A_q(t)\mathbf P,\qquad A_q(t)=D\big[-I+G(t)\hat W(q)\big],\qquad G(t)=\mathrm{diag}\big(F'_e(t),F'_i(t)\big).$$

**4. Floquet theory for each $q$.** Integrate $\dot M=A_q(t)M$, $M(0)=I$, together with the orbit over $[0,T]$ to get $M_T(q)$. Its eigenvalues $\mu_\pm(q)$ are the multipliers, and the mode grows if $\max_q|\mu(q)|>1$. The unstable wavenumber is $q^*=\arg\max_q|\mu(q)|$.
- $\mu>1$ real: spatial pattern $\propto\cos(q^*x)$ that grows with the period of the drive.
- $\mu<-1$ real: the pattern flips sign every period (period $2T$).
- complex pair: a second frequency enters.

**5. Consistency check.** With $A=0$ the orbit is the fixed point, $A_q$ is constant, $M_T=e^{J(q)T}$ and $\mu=e^{\lambda T}$. The code below confirms this, which is a good test of any monodromy code.

In [ ]:
def floquet_modes(qs, A, T, ncyc=30):
    drive = lambda t: np.array([A*np.cos(2*np.pi*t/T), 0.])
    orbit = lambda t, u: (-u + Fn(W@u + h + drive(t))) / tau
    u0 = solve_ivp(orbit, [0, ncyc*T], ustar, rtol=1e-9, atol=1e-11).y[:, -1]            # relax onto the periodic orbit
    Wq = np.array([What(q) for q in qs])                                                   # (nq, 2, 2)
    def rhs(t, y):
        u = y[:2]; arg = W@u + h + drive(t); Gt = np.diag(Fn(arg)*(1-Fn(arg)))
        Aq = D@(-np.eye(2) + Gt@Wq)                                                        # broadcasts over q: (nq, 2, 2)
        return np.concatenate([orbit(t, u), (Aq @ y[2:].reshape(len(qs), 2, 2)).ravel()])
    M0 = np.tile(np.eye(2), (len(qs), 1, 1)).ravel()
    sol = solve_ivp(rhs, [0, T], np.concatenate([u0, M0]), rtol=1e-9, atol=1e-11)
    return np.linalg.eigvals(sol.y[2:, -1].reshape(len(qs), 2, 2)), u0                     # multipliers (nq, 2), orbit at phase 0

qs2 = np.linspace(0.02, 2, 100); T = 6.0
mu0, _ = floquet_modes(qs2, 0., T)                                                         # check: A = 0 gives exp(lambda T)
lam2 = np.array([max(np.linalg.eigvals(Jq(q)), key=lambda z: z.real) for q in qs2])
print("A=0 check, max error in |mu| vs exp(Re lambda T):", abs(abs(mu0).max(1) - np.exp(lam2.real*T)).max())
for A in (0.0, 0.5, 1.5):
    mu, _ = floquet_modes(qs2, A, T); r = abs(mu).max(1); j = r.argmax()
    print(f"A = {A}:  max|mu| = {r[j]:.3f} at q* = {qs2[j]:.2f}  ->", "unstable" if r[j] > 1 else "stable")

**Locating the stability boundary.** To find the drive amplitude where patterns stop growing, track the single number $\rho(A)=\max_q|\mu(q)|$ and find where it crosses 1. It can be bracketed by a coarse scan and then refined by bisection. The same idea gives a boundary in any two parameters by repeating the scan along one of them.

In [ ]:
rho = lambda A: abs(floquet_modes(qs2, A, T)[0]).max()
As = np.linspace(0, 1.5, 7); rhos = [rho(A) for A in As]
print("A     :", As.round(2)); print("rho(A):", np.round(rhos, 3))
lo, hi = [(As[k], As[k+1]) for k in range(len(As)-1) if (rhos[k]-1)*(rhos[k+1]-1) < 0][0]
for _ in range(14):
    mid = (lo+hi)/2; lo, hi = (mid, hi) if (rho(mid)-1)*(rho(lo)-1) > 0 else (lo, mid)
print(f"max|mu| crosses 1 at A = {(lo+hi)/2:.4f}")

### Check the forced theory against a simulation
Start the field on the periodic orbit at phase 0 (period $T$) with a tiny cosine perturbation at $q=2\pi n/L$, then record the Fourier coefficient at that wavenumber once per period. Its ratio from one period to the next should approach the dominant multiplier $\mu(q)$, signed.

In [ ]:
A_d, n = 0.5, 8; q = 2*np.pi*n/L
mu_all, u0 = floquet_modes(np.array([q]), A_d, T)
mu_dom = mu_all[0][abs(mu_all[0]).argmax()].real
u = np.array([u0[0] + 1e-7*np.cos(2*np.pi*n*xg/L), u0[1] + 0*xg]); dt = 0.005; spp = int(round(T/dt)); coef = []
drv = lambda t: np.array([[A_d*np.cos(2*np.pi*t/T)], [0.]])
for s in range(5*spp):
    if s % spp == 0: coef.append(np.fft.fft(u[0])[n].real)
    t = s*dt; u = heun(u, dt, drv(t), drv(t+dt))
coef.append(np.fft.fft(u[0])[n].real)
print(f"A = {A_d}, q = {q:.3f}:  dominant multiplier from Floquet = {mu_dom:.3f}")
print("simulated period-to-period ratios of the mode coefficient:", np.round(np.array(coef[1:])/np.array(coef[:-1]), 3))

## Simulator
Convolution with a periodic Gaussian is a multiplication in Fourier space, so each step is a few FFTs. The state array has shape `(B, *space)`: row `b` is an independent simulation with its own period `T[b]` and amplitude `A[b]`, so many runs can be integrated in one loop. `shape=(N,)` is one spatial dimension and `shape=(N,N)` is two. `snaps` saves $U_e$ at chosen times. `on_step` lets you accumulate any statistic as the run goes. Time stepping is Heun (RK2).

In [ ]:
def kernel_hat(shape, sigma):
    # Fourier transform of a unit-integral Gaussian of width sigma on a periodic grid
    ks = [2*np.pi*np.fft.fftfreq(n) for n in shape[:-1]] + [2*np.pi*np.fft.rfftfreq(shape[-1])]
    k2 = sum(g**2 for g in np.meshgrid(*ks, indexing='ij'))
    return np.exp(-sigma**2*k2/4)

def simulate(T, A, shape, tmax=2000., dt=0.5, gi_over_ge=0., seed=0, snaps=(), on_step=None):
    T = np.atleast_1d(np.asarray(T, float)); A = np.atleast_1d(np.asarray(A, float)); B = T.size
    ax = tuple(range(1, len(shape)+1)); full = (B,)+tuple(shape); bc = (slice(None),)+(None,)*len(shape)
    Ke, Ki = kernel_hat(shape, se)[None], kernel_hat(shape, si)[None]
    conv = lambda K,U: np.fft.irfftn(K*np.fft.rfftn(U, axes=ax), s=shape, axes=ax)
    rng = np.random.default_rng(seed)
    Ue = 0.1*rng.random(full); Ui = 0.1*rng.random(full)        # small random initial condition
    S = lambda t: (A*(np.sin(2*np.pi*t/T) > 0.8))[bc]
    def rhs(Ue, Ui, t):
        ce, ci = conv(Ke,Ue), conv(Ki,Ui); s = S(t)
        return (-Ue+F(aee*ce-aie*ci-the+s))/te, (-Ui+F(aei*ce-aii*ci-thi+gi_over_ge*s))/ti
    out = {}; snap_steps = {int(round(s/dt)):s for s in snaps}
    for n in range(int(round(tmax/dt))):
        t = n*dt
        k1e,k1i = rhs(Ue,Ui,t); k2e,k2i = rhs(Ue+dt*k1e, Ui+dt*k1i, t+dt)
        Ue = Ue + dt/2*(k1e+k2e); Ui = Ui + dt/2*(k1i+k2i)
        if on_step: on_step(n, t+dt, Ue, Ui)
        if n+1 in snap_steps: out[snap_steps[n+1]] = Ue.copy()
    return Ue, Ui, out

## Example call
One run in one spatial dimension: 100 cells, $A=0.8$, period 50 ms, with $U_e$ saved every 5 ms over the last 500 ms. The same function runs in two dimensions with `shape=(N,N)`, and `T` and `A` can be arrays so many settings run in one call.

In [ ]:
times = [2500.+5*k for k in range(100)]
t0 = time.time()
Ue, Ui, snaps = simulate([50.], [0.8], (100,), tmax=3000., snaps=times)
print(f"ran in {time.time()-t0:.2f} s")
H = np.array([snaps[s][0] for s in times])                   # (time, x)
plt.figure(figsize=(4,3.2)); plt.imshow(H, aspect='auto', origin='lower', extent=[0,100,times[0],times[-1]])
plt.xlabel('x (cells)'); plt.ylabel('t (ms)'); plt.colorbar(label='$U_e$'); plt.show()